In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
# Conda environment yolo-train (registered as a Jupyter kernel) with ultralytics,
# torch (+cu121), scikit-learn, joblib, opencv, scipy, pandas, numpy, matplotlib,
# ipywidgets.
"""
conda activate yolo-train
# (all dependencies already present in the yolo-train env)
"""


In [ ]:
# === Cell 3: Imports ===

%matplotlib inline

import os
import sys
import re
import time
import random
import platform
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import joblib
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.ndimage import distance_transform_edt, label as nd_label

from ultralytics import YOLO
import sklearn

import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Plot style ---
_FW            = 16 / 2.54   # full \textwidth in inches
_FONT_SIZE     = 10          # base font size at \textwidth
_YLIM_HEADROOM = 1.18        # headroom above bars for bold value labels

# Sampling-algorithm / source colours (as in the sampling notebook)
C_EDT  = "#2a9d8f"   # EDT_PEAK (teal)
C_GRID = "#e76f51"   # line-scanner grid (orange)
C_GT   = "#264653"   # ground-truth reference

plt.style.use("default")
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         _FONT_SIZE,
    "axes.titlesize":    _FONT_SIZE + 1,
    "axes.labelsize":    _FONT_SIZE,
    "legend.fontsize":   _FONT_SIZE - 1,
    "savefig.dpi":       150,
    "figure.dpi":        110,
    "axes.grid":         True,
    "grid.alpha":        0.25,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "legend.frameon":    False,
    "mathtext.fontset":  "dejavusans",
})

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# === Cell 4: Config ===

PROJECT_NAME = "20260709_Whole_Pipeline_Analysis_Speed"

# --- Input data ---
IMAGES_DIR  = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Combined_Image_Dataset/1080_resolution")
SPECTRA_DIR = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/01_Preprocessing_Investigation/C2_als_z-score")

# --- Ground truth ---
SEG_LABELS_DIR      = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/labels/all")
IR_GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Balanced_NIR_Dataset/spectra_ground_truth_v7.csv")

# Object-level split assignment (cv / test / skipped), shared with the YOLO and IR
# notebooks. Objects in neither cv nor test are treated as "unlisted".
OBJECT_SPLIT_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/object_split.csv")

# --- Evaluation models (test objects held out from training) ---
# YOLO final-training model: trained on fold-0 CV-train, validated on fold-0 CV-val.
SEG_MODEL_PATH = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/20260529_Yolo_Final_Training_yolov8n/"
    "runs/20260529_Yolo_Final_Training_yolov8n_20260529_Yolo_Final_Training_yolov8n/weights/best.pt"
)
# IR benchmark best estimator (LR), refit on the CV pool only (not the all-data
# deployment model).
IR_CLASSIFIER_MODEL_PATH = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/05_IR_Classification/20260629_Classifier_Benchmark_Best_Preprocessing/"
    "results/models/20260629_Classifier_Benchmark_Best_Preprocessing_LR_best_estimator.joblib"
)

# Split used for the whole-pipeline metrics (needs image GT masks and IR GT classes).
# Other splits remain browsable in the widget but contribute no metric rows.
EVAL_SPLIT = "test"

# --- Segmentation matching ---
IOU_MATCH_THRESHOLD = 0.50   # a predicted instance is a TP if its greedy-matched
                             # GT instance has mask IoU >= this value

# --- EDT sampling point (one pole-of-inaccessibility point per instance) ---
# Valid iff the point exists on the predicted valid mask and lies inside the matched
# GT valid mask (for body: GT body and not GT label).
MIN_VALID_AREA_PX2 = 150     # predicted valid area below this -> no EDT point

# --- Grid sampling (line-scanner grid, as in the sampling notebook) ---
# Uses grid spacing and boundary erosion only. Valid iff the number of grid hits inside
# both the predicted and the matched GT valid mask exceeds GRID_MIN_POINTS.
GRID_SPACING_X_MM = 3.5
GRID_SPACING_Y_MM = 3.5
GRID_EROSION_MM   = 1.0
GRID_MIN_POINTS   = 200      # strictly greater-than test

# Physical calibration (px <-> mm), from the sampling notebook.
CALIB_PIXELS  = 355          # pixels measured for the reference length
CALIB_CM      = 10.0         # physical reference length [cm]
PIXELS_PER_CM = CALIB_PIXELS / CALIB_CM        # -> 35.5 px/cm
PIXELS_PER_MM = PIXELS_PER_CM / 10.0           # -> 3.55 px/mm

# --- YOLO inference defaults ---
CONF_THRESHOLD = 0.50
IOU_THRESHOLD  = 0.50        # YOLO NMS IoU (distinct from IOU_MATCH_THRESHOLD)
IMAGE_SIZE     = 640
MAX_DET        = 300
DEVICE         = 0 if torch.cuda.is_available() else "cpu"

# --- Dataset semantics ---
CLASS_MAP = {          # YOLO class id -> region name (training convention)
    "body":  0,
    "cap":   1,
    "label": 2,
}
CLASS_ID_TO_NAME = {v: k for k, v in CLASS_MAP.items()}
VALID_REGIONS    = ["body", "cap", "label"]

# Ground-truth CSV columns holding the true material class per region.
REGION_TO_GT_COLUMN = {
    "body":  "body_class",
    "label": "label_class",
    "cap":   "cap_class",
}

# Filename patterns
IMAGE_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_per_(?P<per>\d+)(?:_.*)?$",
    re.IGNORECASE,
)
SPECTRUM_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)(?:_per_(?P<per>\d+))?_(?P<region>body|cap|label)(?:_.*)?$",
    re.IGNORECASE,
)

# --- Output ---
OUTPUT_DIR  = Path("./20260709_Whole_Pipeline_Analysis")
RESULTS_DIR = OUTPUT_DIR / "results_speed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Reproducibility ---
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


In [ ]:
# === Cell 5: Environment and Path Checks ===

print("Environment")
print("-" * 80)
print(f"Python:              {sys.version.split()[0]}")
print(f"Platform:            {platform.platform()}")
print(f"torch:               {torch.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device:         {torch.cuda.get_device_name(0)}")
print(f"scikit-learn:        {sklearn.__version__}")
print(f"OpenCV:              {cv2.__version__}")
print(f"NumPy:               {np.__version__}")

print("\nPath checks")
print("-" * 80)
_paths = {
    "IMAGES_DIR":               IMAGES_DIR,
    "SPECTRA_DIR":              SPECTRA_DIR,
    "SEG_LABELS_DIR":           SEG_LABELS_DIR,
    "IR_GROUND_TRUTH_CSV":      IR_GROUND_TRUTH_CSV,
    "OBJECT_SPLIT_CSV":         OBJECT_SPLIT_CSV,
    "SEG_MODEL_PATH":           SEG_MODEL_PATH,
    "IR_CLASSIFIER_MODEL_PATH": IR_CLASSIFIER_MODEL_PATH,
}
_all_ok = True
for name, p in _paths.items():
    ok = p.exists()
    _all_ok &= ok
    print(f"  [{'OK ' if ok else 'MISSING'}] {name:26s} {p}")
if not _all_ok:
    print("\nWARNING: one or more required paths are missing — fix before running Cell 6.")
else:
    print("\nAll required paths resolved.")


In [ ]:
# === Cell 6: Assets, Splits and Models ===

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# --- Image pool: image_pool[object_key][per_idx] = Path ---
image_pool = defaultdict(dict)
image_skipped = []
for p in sorted(IMAGES_DIR.iterdir()):
    if not (p.is_file() and p.suffix.lower() in IMAGE_EXTS):
        continue
    m = IMAGE_REGEX.match(p.stem)
    if not m:
        image_skipped.append(p.name)
        continue
    object_key = f"{m.group('material')}_obj_{m.group('object_id')}"
    per_idx    = int(m.group("per"))
    image_pool[object_key][per_idx] = p

image_pool = {k: dict(sorted(v.items())) for k, v in sorted(image_pool.items())}
n_images   = sum(len(v) for v in image_pool.values())
print(f"Indexed {n_images} image(s) across {len(image_pool)} object(s) in {IMAGES_DIR}")
if image_skipped:
    print(f"  {len(image_skipped)} file(s) did not match the image pattern "
          f"(showing up to 5): {image_skipped[:5]}")
if not image_pool:
    raise FileNotFoundError(f"No pattern-matching images found in {IMAGES_DIR}")

# --- Spectrum index: spectra_index[(object_key, region)] = Path (per object) ---
spectra_index = {}
spectra_skipped = []
for p in sorted(SPECTRA_DIR.iterdir()):
    if not p.is_file():
        continue
    m = SPECTRUM_REGEX.match(p.name)
    if not m:
        spectra_skipped.append(p.name)
        continue
    object_key = f"{m.group('material')}_obj_{m.group('object_id')}"
    region     = m.group("region").lower()
    spectra_index[(object_key, region)] = p
print(f"\nIndexed {len(spectra_index)} spectrum file(s) in {SPECTRA_DIR}")
if spectra_skipped:
    print(f"  {len(spectra_skipped)} file(s) did not match the spectrum pattern "
          f"(showing up to 5): {spectra_skipped[:5]}")

# --- Split assignment: object_key -> "cv" / "test" / "unlisted" ---
# object_split.csv rows are matched on (upper-case material, int object_id); pooled
# objects that are missing or "skipped" are labelled "unlisted".
def _obj_key_parts(object_key):
    m = re.match(r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)$", object_key)
    if not m:
        return None
    return m.group("material").upper(), int(m.group("object_id"))

split_lookup = {}   # (MATERIAL, int_id) -> split
if OBJECT_SPLIT_CSV.exists():
    _split_df = pd.read_csv(OBJECT_SPLIT_CSV)
    _split_df.columns = [c.strip().lower() for c in _split_df.columns]
    _mat_col = "material" if "material" in _split_df.columns else _split_df.columns[0]
    _id_col  = "object_id" if "object_id" in _split_df.columns else _split_df.columns[1]
    _spl_col = "split" if "split" in _split_df.columns else _split_df.columns[-1]
    for _, row in _split_df.iterrows():
        try:
            key = (str(row[_mat_col]).strip().upper(), int(row[_id_col]))
        except (ValueError, TypeError):
            continue
        split_lookup[key] = str(row[_spl_col]).strip().lower()
    print(f"\nLoaded object split: {len(split_lookup)} object(s) from {OBJECT_SPLIT_CSV.name}")
else:
    print(f"\nWARNING: {OBJECT_SPLIT_CSV} not found — every object will be 'unlisted'.")

object_split = {}   # object_key -> normalised split
for object_key in image_pool:
    parts = _obj_key_parts(object_key)
    raw = split_lookup.get(parts, "unlisted") if parts else "unlisted"
    object_split[object_key] = raw if raw in ("cv", "test") else "unlisted"

_split_counts = pd.Series(object_split).value_counts().to_dict()
print(f"  Split distribution over pooled objects: {_split_counts}")
eval_object_keys = sorted(k for k, s in object_split.items() if s == EVAL_SPLIT)
print(f"  Objects in EVAL_SPLIT='{EVAL_SPLIT}': {len(eval_object_keys)}")
if not eval_object_keys:
    print("  WARNING: no objects in the evaluation split — the metrics cells will be empty.")

# --- Evaluation models (held out) ---
print(f"\nLoading YOLO (held-out) segmentation model: {SEG_MODEL_PATH.name}")
yolo_model_eval = YOLO(str(SEG_MODEL_PATH))

print(f"Loading IR (held-out) classifier:            {IR_CLASSIFIER_MODEL_PATH.name}")
ir_classifier_eval = joblib.load(IR_CLASSIFIER_MODEL_PATH)

# Models pickled under sklearn 1.8.0 lack LogisticRegression's `multi_class`
# attribute, which predict_proba on older sklearn runtimes reads; set it to "auto".
def _patch_missing_multi_class(estimator):
    step = estimator.steps[-1][1] if hasattr(estimator, "steps") else estimator
    if type(step).__name__ == "LogisticRegression" and not hasattr(step, "multi_class"):
        step.multi_class = "auto"
        print(f"  Patched missing 'multi_class' on {type(step).__name__} "
              f"(sklearn {sklearn.__version__} runtime vs. newer pickling version).")
_patch_missing_multi_class(ir_classifier_eval)

try:
    _clf_step = ir_classifier_eval.steps[-1][1]
    _clf_name = type(_clf_step).__name__
except AttributeError:
    _clf_name = type(ir_classifier_eval).__name__
ir_classes = list(getattr(ir_classifier_eval, "classes_", []))
print(f"  Classifier type:    {_clf_name}")
print(f"  Known classes:      {ir_classes}")

# --- IR ground-truth CSV ---
ir_gt_df = pd.read_csv(IR_GROUND_TRUTH_CSV, sep=";")
_gt_filename_col = "filename" if "filename" in ir_gt_df.columns else ir_gt_df.columns[0]
print(f"\nLoaded IR ground truth: {len(ir_gt_df)} row(s)")
print(f"  Columns: {list(ir_gt_df.columns)}")
missing_gt_cols = [c for c in REGION_TO_GT_COLUMN.values() if c not in ir_gt_df.columns]
if missing_gt_cols:
    print(f"  NOTE: expected GT class column(s) not found: {missing_gt_cols}")

print(f"\nReady: {len(image_pool)} object(s) / {n_images} image(s); "
      f"{len(eval_object_keys)} eval object(s) in '{EVAL_SPLIT}'.")


In [ ]:
# === Cell 7: Pipeline Helper Functions ===

# --- Colour palettes (same region scheme as the sampling / segmentation notebooks) ---
CLASS_COLORS_RGB = {          # prediction colours (solid, vivid)
    0: (255,   0,   0),       # body  - red
    1: (  0, 200,   0),       # cap   - green
    2: (  0,  80, 255),       # label - blue
}
CLASS_COLORS_MPL = {cid: tuple(c / 255.0 for c in rgb)
                    for cid, rgb in CLASS_COLORS_RGB.items()}
GT_COLORS_RGB = {0: (255, 160, 160), 1: (160, 255, 160), 2: (160, 200, 255)}
DEFAULT_COLOR_RGB = (255, 255, 0)
MASK_ALPHA = 0.35

def _rgb2bgr(c):   return (c[2], c[1], c[0])
def _cls_bgr(cid): return _rgb2bgr(CLASS_COLORS_RGB.get(int(cid), DEFAULT_COLOR_RGB))

# --- IR: spectrum loading and classification ---
def load_spectrum(path):
    """Load a headerless wavenumber;reflectance file; return the reflectance vector
    sorted by ascending wavenumber (the classifier's feature layout)."""
    df = pd.read_csv(path, sep=";", header=None,
                     names=["wavenumber", "reflectance"], engine="python")
    df["wavenumber"]  = pd.to_numeric(df["wavenumber"],  errors="coerce")
    df["reflectance"] = pd.to_numeric(df["reflectance"], errors="coerce")
    df = df.dropna().sort_values("wavenumber").reset_index(drop=True)
    if df.empty:
        raise ValueError(f"Empty / non-numeric spectrum: {path}")
    return df["reflectance"].to_numpy(dtype=np.float64)

def classify_region(object_key, region):
    """Classify the (object, region) spectrum with the held-out IR model.

    Confidence is the maximum predict_proba value (the `multi_class` patch in the
    Assets cell keeps this working). Returns (label, confidence, spectrum_path):
      spectrum found      -> (predicted_class, max_proba_or_None, Path)
      no spectrum on disk -> ("unknown", None, None)
      load/predict error  -> ("error", None, Path)
    """
    path = spectra_index.get((object_key, region))
    if path is None:
        return "unknown", None, None
    try:
        vec = load_spectrum(path).reshape(1, -1)
        pred = str(ir_classifier_eval.predict(vec)[0])
        conf = None
        if hasattr(ir_classifier_eval, "predict_proba"):
            conf = float(np.max(ir_classifier_eval.predict_proba(vec)[0]))
        return pred, conf, path
    except Exception as exc:
        print(f"  IR classify error for {path.name}: {exc}")
        return "error", None, path

def lookup_ir_gt(object_key, region):
    """Return the GT material class for object+region, or 'unknown'."""
    col = REGION_TO_GT_COLUMN.get(region)
    if col is None or col not in ir_gt_df.columns:
        return "unknown"
    row = ir_gt_df[ir_gt_df[_gt_filename_col] == object_key]
    if row.empty:
        return "unknown"
    val = row.iloc[0][col]
    if pd.isna(val) or str(val).strip() == "":
        return "unknown"
    return str(val).strip()

# --- Mask rasterisation and per-instance valid masks ---
# Body valid mask = body AND NOT union(label masks), on both predicted and GT side.
def rasterise_polygon(poly_px, h, w):
    """Rasterise an (N,2) pixel polygon to a boolean (h, w) mask."""
    mask = np.zeros((h, w), dtype=np.uint8)
    if poly_px is None or len(poly_px) < 3:
        return mask.astype(bool)
    pts = np.asarray(poly_px, dtype=np.int32).reshape(-1, 1, 2)
    cv2.fillPoly(mask, [pts], 1)
    return mask.astype(bool)

def build_masks(instances, h, w):
    """Build per-region mask entries from instances {class_id, polygon_px[, conf]}.

    Returns per-region lists of {raw, valid, polygon_px, conf} plus 'label_union';
    order within a class is preserved, so indices are stable for matching."""
    body_insts  = [i for i in instances if i["class_id"] == 0]
    cap_insts   = [i for i in instances if i["class_id"] == 1]
    label_insts = [i for i in instances if i["class_id"] == 2]

    label_masks_raw = [rasterise_polygon(i["polygon_px"], h, w) for i in label_insts]
    label_union = np.zeros((h, w), dtype=bool)
    for lm in label_masks_raw:
        label_union |= lm

    body_entries = []
    for inst in body_insts:
        raw = rasterise_polygon(inst["polygon_px"], h, w)
        body_entries.append({"raw": raw, "valid": raw & ~label_union,
                             "polygon_px": inst["polygon_px"],
                             "conf": inst.get("conf", np.nan)})
    cap_entries = [{"raw": rasterise_polygon(i["polygon_px"], h, w),
                    "valid": rasterise_polygon(i["polygon_px"], h, w),
                    "polygon_px": i["polygon_px"], "conf": i.get("conf", np.nan)}
                   for i in cap_insts]
    label_entries = [{"raw": raw, "valid": raw.copy(), "polygon_px": inst["polygon_px"],
                      "conf": inst.get("conf", np.nan)}
                     for inst, raw in zip(label_insts, label_masks_raw)]
    return {"body": body_entries, "cap": cap_entries, "label": label_entries,
            "label_union": label_union}

def _crop_to_bbox(mask: np.ndarray):
    """Crop a boolean mask to its tight bounding box with border-aware padding.

    Returns (padded_crop, row_offset, col_offset). The padded crop reproduces
    full-frame distance_transform_edt results exactly: interior sides get a
    1 px False ring (background), image-border sides stay un-padded. Full-image
    coordinates are recovered as (x + col_offset, y + row_offset).
    """
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any():
        return mask, 0, 0
    r0 = int(np.argmax(rows))
    r1 = int(len(rows) - 1 - np.argmax(rows[::-1]))
    c0 = int(np.argmax(cols))
    c1 = int(len(cols) - 1 - np.argmax(cols[::-1]))
    H, W = mask.shape
    crop = mask[r0:r1 + 1, c0:c1 + 1]
    pt_top    = 1 if r0 > 0     else 0
    pt_bottom = 1 if r1 < H - 1 else 0
    pt_left   = 1 if c0 > 0     else 0
    pt_right  = 1 if c1 < W - 1 else 0
    padded = np.pad(
        crop, ((pt_top, pt_bottom), (pt_left, pt_right)),
        mode="constant", constant_values=False,
    )
    return padded, r0 - pt_top, c0 - pt_left


def compute_edt_peak(binary_mask):
    """Return (x, y, distance) of the EDT peak (pole of inaccessibility) of a mask.

    Handles holes/annular shapes; (None, None, 0.0) for an empty mask. Uses the
    bbox crop, which gives the same result as a full-frame EDT."""
    if binary_mask is None or binary_mask.sum() == 0:
        return None, None, 0.0
        crop, r_off, c_off = _crop_to_bbox(binary_mask)
        edt = distance_transform_edt(crop)
        r, c = np.unravel_index(int(np.argmax(edt)), edt.shape)
        return int(c) + c_off, int(r) + r_off, float(edt[r, c])

# --- Greedy IoU matching (per class): predicted <-> GT instances ---
def mask_iou(a, b):
    """IoU of two boolean masks."""
    inter = np.logical_and(a, b).sum()
    if inter == 0:
        return 0.0
    return float(inter) / float(np.logical_or(a, b).sum())

def greedy_match(pred_masks, gt_masks, thr):
    """Greedy highest-IoU-first assignment. Returns:
      pred_to_gt    : {pred_idx -> gt_idx}   (only pairs with IoU >= thr; TP set)
      pred_best_iou : {pred_idx -> best IoU with ANY gt}
      pred_best_gt  : {pred_idx -> gt_idx achieving that best IoU, or None if 0}
      gt_matched    : set of matched gt_idx (threshold-gated)
    pred_best_gt is independent of `thr`, so sampling-point validity can be checked
    against the overlapping GT region even for `duplicate` detections; TP/FP/FN
    stay threshold-gated.
    """
    n_p, n_g = len(pred_masks), len(gt_masks)
    pred_best_iou = {pi: 0.0 for pi in range(n_p)}
    pred_best_gt  = {pi: None for pi in range(n_p)}
    pairs = []
    for pi in range(n_p):
        for gi in range(n_g):
            iou = mask_iou(pred_masks[pi], gt_masks[gi])
            if iou > pred_best_iou[pi]:
                pred_best_iou[pi] = iou
                pred_best_gt[pi]  = gi
            if iou > 0:
                pairs.append((iou, pi, gi))
    pairs.sort(key=lambda t: t[0], reverse=True)
    pred_to_gt, gt_matched = {}, set()
    for iou, pi, gi in pairs:
        if iou < thr:
            break                       # sorted desc -> no further pair qualifies
        if pi in pred_to_gt or gi in gt_matched:
            continue
        pred_to_gt[pi] = gi
        gt_matched.add(gi)
    return pred_to_gt, pred_best_iou, pred_best_gt, gt_matched

# --- Line-scanner grid scan (as in the sampling notebook) ---
def grid_scan(mask_data, spacing_x, spacing_y, erosion_px, img_w, img_h):
    """Classify every grid point against the valid instance masks.

    A point is a hit for an instance iff it lies in exactly one valid instance mask
    (across all classes). Optional boundary erosion keeps EDT >= erosion_px on the
    valid mask. Returns 'hits' [(x, y, cls, inst)], 'counts' {(cls, inst): n} and
    the grid coordinate arrays."""
    grid_xs = np.arange(0, img_w, spacing_x)
    grid_ys = np.arange(0, img_h, spacing_y)
    mask_entries = []   # (cls_name, inst_idx, test_mask)
    for cls_name in ("body", "cap", "label"):
        for inst_idx, entry in enumerate(mask_data[cls_name]):
            vmask = entry["valid"]
            if erosion_px > 0:
                crop, r_off, c_off = _crop_to_bbox(vmask)
                eroded_crop = distance_transform_edt(crop) >= erosion_px
                test_mask = np.zeros_like(vmask)
                ch, cw = eroded_crop.shape
                test_mask[r_off:r_off + ch, c_off:c_off + cw] = eroded_crop
            else:
                test_mask = vmask
            mask_entries.append((cls_name, inst_idx, test_mask))
    hits, counts = [], defaultdict(int)
    for y in grid_ys:
        for x in grid_xs:
            matched = [(cn, ii) for cn, ii, vm in mask_entries if vm[y, x]]
            if len(matched) == 1:
                cn, ii = matched[0]
                hits.append((int(x), int(y), cn, ii))
                counts[(cn, ii)] += 1
    return {"grid_xs": grid_xs, "grid_ys": grid_ys,
            "hits": hits, "counts": dict(counts)}

# --- Instance extraction: YOLO result / GT txt -> instances ---
def yolo_result_to_instances(result):
    """Convert an Ultralytics result to a list of {class_id, conf, polygon_px}."""
    instances = []
    if result.boxes is None or len(result.boxes) == 0:
        return instances
    cls_ids = result.boxes.cls.cpu().numpy().astype(int)
    confs   = result.boxes.conf.cpu().numpy()
    polys   = result.masks.xy if (result.masks is not None and result.masks.xy is not None) else []
    for i in range(len(cls_ids)):
        poly = polys[i] if i < len(polys) else None
        instances.append({"class_id": int(cls_ids[i]), "conf": float(confs[i]),
                          "polygon_px": None if poly is None else np.asarray(poly)})
    return instances

def load_gt_instances(image_stem, h, w):
    """Read the YOLO-seg GT .txt for an image stem and return instances in pixel
    space, or None if no GT file exists (e.g. unlisted objects)."""
    label_path = SEG_LABELS_DIR / (image_stem + ".txt")
    if not label_path.exists():
        return None
    instances = []
    with label_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cid = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]
            poly = np.array([[cx * w, cy * h]
                             for cx, cy in zip(coords[0::2], coords[1::2])], dtype=np.float64)
            if len(poly) >= 3:
                instances.append({"class_id": cid, "polygon_px": poly})
    return instances


In [ ]:
# === Cell 8: Per-Image Pipeline Analysis ===

# Grid spacing / erosion in pixels (mm controls -> px), computed once.
GRID_SX_PX = max(1, int(round(GRID_SPACING_X_MM * PIXELS_PER_MM)))
GRID_SY_PX = max(1, int(round(GRID_SPACING_Y_MM * PIXELS_PER_MM)))
GRID_ER_PX = int(round(GRID_EROSION_MM * PIXELS_PER_MM))

# Per-object IR results are perspective-independent and cached.
_ir_cache = {}
def ir_result_for(object_key, region):
    """Cached IR prediction, GT class and correctness (Y/N/NA) for one object region."""
    key = (object_key, region)
    if key not in _ir_cache:
        pred, conf, spath = classify_region(object_key, region)
        gt = lookup_ir_gt(object_key, region)
        if pred in ("unknown", "error") or gt == "unknown":
            correct = "NA"
        else:
            correct = "Y" if pred == gt else "N"
        _ir_cache[key] = {"ir_pred": pred, "ir_conf": conf, "ir_gt": gt,
                          "ir_correct": correct, "spectrum_found": spath is not None}
    return _ir_cache[key]


# --- Combining logic ---
# Used only by the Aggregation cell and the widget; never written to the raw CSVs.
def effective_ir_status(ir_correct, sampled):
    """Effective IR outcome with sampling availability folded in:
      no GT class        -> "NA" (ungradeable, independent of sampling)
      sampling invalid   -> "--" (material unobtainable in a live setting)
      sampling valid     -> ir_correct ("Y"/"N")
    """
    if ir_correct == "NA":
        return "NA"
    if not sampled:
        return "--"
    return ir_correct

def pipeline_bucket(detected, sampled, ir_correct):
    """Mutually exclusive loss/success bucket for one (perspective, region).

    Loss is attributed to the first failing stage (segmentation -> sampling -> IR).
    Only valid for region-perspectives where the region is present (n_gt >= 1)."""
    if not detected:
        return "no_detection"
    if not sampled:
        return "no_sample"
    if ir_correct == "Y":
        return "success"
    if ir_correct == "N":
        return "ir_wrong"
    return "no_ir_gt"       # detected and sampled, but no IR GT class


def analyze_image(image_path, object_key, split, return_detail=False):
    """Run the held-out YOLO + IR pipeline on one image and score it against GT.

    Returns (instance_records, region_records, detail):
      instance_records : one row per PREDICTED instance (segmentation match +
                         sampling validity + IR result of its region)
      region_records   : one row per (perspective, region) with seg TP/FP/FN,
                         duplicate count, region-level sampling/IR/pipeline flags
      detail           : dict for rendering (masks, grid, per-instance draw data)
                         (only populated when return_detail=True)
    """
    per_idx = None
    m = IMAGE_REGEX.match(Path(image_path).stem)
    if m:
        per_idx = int(m.group("per"))

    # --- YOLO inference (held-out model) ---
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    result = yolo_model_eval.predict(
        source=str(image_path), conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
        imgsz=IMAGE_SIZE, device=DEVICE, retina_masks=True, max_det=MAX_DET,
        rect=True, verbose=False, save=False,
    )[0]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    inference_ms = (time.perf_counter() - t0) * 1000.0

    h, w = result.orig_img.shape[:2]
    pred_instances = yolo_result_to_instances(result)
    pred_mask_data = build_masks(pred_instances, h, w)

    gt_instances = load_gt_instances(Path(image_path).stem, h, w)
    has_gt = gt_instances is not None
    gt_mask_data = build_masks(gt_instances, h, w) if has_gt else None

    # --- grid scan on predicted masks (once per image) ---
    grid = grid_scan(pred_mask_data, GRID_SX_PX, GRID_SY_PX, GRID_ER_PX, w, h)

    instance_records, region_records = [], []
    draw_instances = []   # for the widget

    for region in VALID_REGIONS:
        pred_entries = pred_mask_data[region]
        gt_entries   = gt_mask_data[region] if has_gt else []
        pred_masks   = [e["raw"] for e in pred_entries]
        gt_masks     = [e["raw"] for e in gt_entries]

        if has_gt:
            pred_to_gt, pred_best_iou, pred_best_gt, gt_matched = greedy_match(
                pred_masks, gt_masks, IOU_MATCH_THRESHOLD)
        else:
            pred_to_gt = {}
            pred_best_iou = {pi: np.nan for pi in range(len(pred_masks))}
            pred_best_gt  = {pi: None   for pi in range(len(pred_masks))}
            gt_matched = set()

        ir = ir_result_for(object_key, region)

        n_gt   = len(gt_masks) if has_gt else np.nan
        n_pred = len(pred_masks)
        tp     = len(pred_to_gt) if has_gt else np.nan
        fp     = (n_pred - tp) if has_gt else np.nan
        fn     = (n_gt - tp) if has_gt else np.nan
        n_dup  = 0

        # Region-level raw signals (no combining logic)
        region_edt_valid_any  = False   # any instance yields a GT-valid EDT point
        region_grid_valid_any = False   # any instance yields a GT-valid grid count
        region_detected       = False   # any predicted instance overlaps a GT region

        for pi, entry in enumerate(pred_entries):
            best_iou = pred_best_iou.get(pi, np.nan)
            if not has_gt:
                match_type = "no_gt"
            elif pi in pred_to_gt:
                match_type = "matched"
            elif best_iou > 0:
                match_type = "duplicate"
            else:
                match_type = "spurious"
            if match_type == "duplicate":
                n_dup += 1
            if match_type in ("matched", "duplicate"):
                region_detected = True

            # Sampling points are tested against the best-overlapping GT instance
            # (independent of the TP threshold); `spurious` instances have none.
            gi_tp   = pred_to_gt.get(pi)                    # threshold-gated (written to CSV)
            gi_best = pred_best_gt.get(pi)                  # threshold-independent
            gt_valid = gt_entries[gi_best]["valid"] if (gi_best is not None) else None

            # --- EDT sampling point ---
            pred_valid = entry["valid"]
            area = int(pred_valid.sum())
            if area >= MIN_VALID_AREA_PX2:
                sx, sy, edt_score = compute_edt_peak(pred_valid)
            else:
                sx, sy, edt_score = None, None, 0.0
            edt_in_pred = sx is not None
            if edt_in_pred and gt_valid is not None:
                edt_in_gt = bool(gt_valid[sy, sx])
            else:
                edt_in_gt = False
            edt_valid = bool(edt_in_pred and (gt_valid is not None) and edt_in_gt)

            # --- grid sampling: predicted hits, and hits inside best-overlap GT ---
            inst_hits = [(x, y) for (x, y, cn, ii) in grid["hits"]
                         if cn == region and ii == pi]
            n_grid_pred = len(inst_hits)
            if gt_valid is not None:
                n_grid_in_gt = sum(1 for (x, y) in inst_hits if gt_valid[y, x])
            else:
                n_grid_in_gt = 0
            grid_valid = bool((gt_valid is not None) and (n_grid_in_gt > GRID_MIN_POINTS))

            # Raw region rollup: any detected instance (matched or duplicate) with a
            # GT-valid sample counts; TP-only gating is applied in the Aggregation cell.
            if match_type in ("matched", "duplicate"):
                region_edt_valid_any  |= edt_valid
                region_grid_valid_any |= grid_valid

            instance_records.append({
                "object_key": object_key, "split": split, "perspective": per_idx,
                "region": region, "pred_instance_idx": pi, "match_type": match_type,
                "iou_best": best_iou,
                "gt_instance_idx": (int(gi_tp) if gi_tp is not None else np.nan),
                "gt_best_overlap_idx": (int(gi_best) if gi_best is not None else np.nan),
                "conf": float(entry.get("conf", np.nan)),
                "pred_valid_area_px2": area,
                "edt_x": (int(sx) if sx is not None else np.nan),
                "edt_y": (int(sy) if sy is not None else np.nan),
                "edt_score": round(float(edt_score), 2),
                "edt_in_pred": edt_in_pred, "edt_in_gt": edt_in_gt, "edt_valid": edt_valid,
                "n_grid_pred": n_grid_pred, "n_grid_in_gt": n_grid_in_gt, "grid_valid": grid_valid,
                "ir_pred": ir["ir_pred"], "ir_conf": ir["ir_conf"],
                "ir_gt": ir["ir_gt"], "ir_correct": ir["ir_correct"],
            })

            draw_instances.append({
                "region": region, "pred_idx": pi, "match_type": match_type,
                "polygon_px": entry["polygon_px"], "edt_xy": (sx, sy) if edt_in_pred else None,
                "edt_score": float(edt_score), "edt_valid": edt_valid, "grid_valid": grid_valid,
                "n_grid_pred": n_grid_pred, "n_grid_in_gt": n_grid_in_gt,
                "iou_best": best_iou,
            })

        # --- Region-level raw record (per-step facts only) ---
        # Pipeline success and effective IR are derived in Aggregation and the widget.
        seg_ok = bool(has_gt and (tp >= 1))
        region_records.append({
            "object_key": object_key, "split": split, "perspective": per_idx,
            "region": region, "has_gt": has_gt,
            "n_gt": n_gt, "n_pred": n_pred, "tp": tp, "fp": fp, "fn": fn,
            "n_duplicates": n_dup, "seg_ok": seg_ok, "detected": region_detected,
            "sampling_edt_ok": region_edt_valid_any, "sampling_grid_ok": region_grid_valid_any,
            "ir_pred": ir["ir_pred"], "ir_conf": ir["ir_conf"],
            "ir_gt": ir["ir_gt"], "ir_correct": ir["ir_correct"],
        })

    detail = {}
    if return_detail:
        detail = {"h": h, "w": w, "result": result,
                  "pred_mask_data": pred_mask_data, "gt_mask_data": gt_mask_data,
                  "has_gt": has_gt, "grid": grid, "draw_instances": draw_instances,
                  "inference_ms": inference_ms, "per_idx": per_idx}
    return instance_records, region_records, detail

print(f"Grid: {GRID_SPACING_X_MM}x{GRID_SPACING_Y_MM} mm -> {GRID_SX_PX}x{GRID_SY_PX} px, "
      f"erosion {GRID_EROSION_MM} mm -> {GRID_ER_PX} px.  analyze_image() ready.")


In [ ]:
# === Cell 9: Batch Analysis ===
# Runs analyze_image on every image of EVAL_SPLIT and saves the raw tables.

_ir_cache.clear()   # clear IR cache before the run

all_instance_records, all_region_records = [], []
_n_obj = len(eval_object_keys)
if _n_obj == 0:
    print(f"No objects in EVAL_SPLIT='{EVAL_SPLIT}'. Nothing to analyse.")
else:
    _t_start = time.perf_counter()
    _img_done = 0
    for oi, object_key in enumerate(eval_object_keys, 1):
        per_map = image_pool.get(object_key, {})
        for per_idx, image_path in per_map.items():
            inst_recs, reg_recs, _ = analyze_image(
                image_path, object_key, object_split[object_key], return_detail=False)
            all_instance_records.extend(inst_recs)
            all_region_records.extend(reg_recs)
            _img_done += 1
        print(f"  [{oi}/{_n_obj}] {object_key}: {len(per_map)} perspective(s)")
    _elapsed = time.perf_counter() - _t_start
    print(f"\nProcessed {_img_done} image(s) across {_n_obj} object(s) in {_elapsed:.1f}s.")

instance_df = pd.DataFrame(all_instance_records)
region_df   = pd.DataFrame(all_region_records)

# --- IR is per object region (perspective-independent): deduplicate ---
if not region_df.empty:
    ir_object_df = (region_df[["object_key", "split", "region",
                               "ir_pred", "ir_conf", "ir_gt", "ir_correct"]]
                    .drop_duplicates(["object_key", "region"])
                    .sort_values(["object_key", "region"])
                    .reset_index(drop=True))
else:
    ir_object_df = pd.DataFrame(
        columns=["object_key", "split", "region", "ir_pred", "ir_conf", "ir_gt", "ir_correct"])

# --- persist raw tables ---
_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
inst_csv = RESULTS_DIR / f"{PROJECT_NAME}_instance_metrics_{_ts}.csv"
reg_csv  = RESULTS_DIR / f"{PROJECT_NAME}_region_metrics_{_ts}.csv"
ir_csv   = RESULTS_DIR / f"{PROJECT_NAME}_ir_object_region_{_ts}.csv"
instance_df.to_csv(inst_csv, index=False)
region_df.to_csv(reg_csv, index=False)
ir_object_df.to_csv(ir_csv, index=False)

# Paths used by the Aggregation cell to reload the tables from disk.
LATEST_INSTANCE_CSV = inst_csv
LATEST_REGION_CSV   = reg_csv
LATEST_IR_CSV       = ir_csv

print(f"\nSaved:")
print(f"  instance-level : {inst_csv.name}  ({len(instance_df)} rows)")
print(f"  region-level   : {reg_csv.name}  ({len(region_df)} rows)")
print(f"  IR object-region: {ir_csv.name}  ({len(ir_object_df)} rows)")

if not instance_df.empty:
    print("\nInstance-level preview:")
    display(instance_df.head(8))


In [ ]:
# === Cell 10: Aggregation and Combining Logic ===
# Derives effective IR, pipeline success and loss attribution from the raw
# region/instance CSVs.

_region = pd.read_csv(LATEST_REGION_CSV)
_inst   = pd.read_csv(LATEST_IR_CSV)         # object-region IR
_instm  = pd.read_csv(LATEST_INSTANCE_CSV)   # per-instance

def _rate(numer, denom):
    return (100.0 * numer / denom) if denom else np.nan

# Bucket display order (stack bottom -> top) and colours
BUCKET_ORDER  = ["success", "ir_wrong", "no_sample", "no_detection", "no_ir_gt"]
BUCKET_LABELS = {"success": "Success", "ir_wrong": "Classification Loss",
                 "no_sample": "Sampling Loss",
                 "no_detection": "Segmentation Loss", "no_ir_gt": "No IR GT"}
BUCKET_COLORS = {"success": C_EDT, "ir_wrong": "#c44e52", "no_sample": C_GRID,
                 "no_detection": "#4a4a4a", "no_ir_gt": "#c7c7c7"}

if _region.empty:
    print("No region rows — run Cell 9 first.")
else:
    # --- 1. Segmentation TP/FP/FN (threshold-gated) ---
    seg = (_region.groupby("region")[["tp", "fp", "fn", "n_gt", "n_pred", "n_duplicates"]]
           .sum().reindex(VALID_REGIONS))
    seg["precision"] = seg["tp"] / (seg["tp"] + seg["fp"]).replace(0, np.nan)
    seg["recall"]    = seg["tp"] / (seg["tp"] + seg["fn"]).replace(0, np.nan)
    seg["f1"] = 2 * seg["precision"] * seg["recall"] / (seg["precision"] + seg["recall"])
    print(f"Segmentation (IoU >= {IOU_MATCH_THRESHOLD}) — summed over eval split, per region")
    print("-" * 78)
    print(seg.round(3).to_string())

    # --- 2. Sampling validity among matched (TP) instances ---
    for _c in ("edt_valid", "grid_valid"):
        _instm[_c] = _instm[_c].map({True: True, False: False, "True": True, "False": False}).fillna(False)
    matched_inst = _instm[_instm["match_type"] == "matched"]
    print("\nSampling validity among matched (TP) instances, per region  [IoU-gated]")
    print("-" * 78)
    if matched_inst.empty:
        print("  (no detected instances)")
        samp = pd.DataFrame()
    else:
        samp = matched_inst.groupby("region").agg(
            n_matched=("edt_valid", "size"),
            edt_valid=("edt_valid", "sum"),
            grid_valid=("grid_valid", "sum"),
            mean_grid_in_gt=("n_grid_in_gt", "mean"),
        ).reindex(VALID_REGIONS)
        samp["edt_valid_pct"]  = 100.0 * samp["edt_valid"]  / samp["n_matched"]
        samp["grid_valid_pct"] = 100.0 * samp["grid_valid"] / samp["n_matched"]
        print(samp.round(2).to_string())

    # --- 3. IR classification (raw, per object region) ---
    print("\nIR classification (raw, per object-region), per region")
    print("-" * 78)
    ir_tab = _inst.groupby("region")["ir_correct"].value_counts().unstack(fill_value=0)
    for col in ("Y", "N", "NA"):
        if col not in ir_tab.columns:
            ir_tab[col] = 0
    ir_tab = ir_tab.reindex(VALID_REGIONS)[["Y", "N", "NA"]]
    ir_tab["assessable"] = ir_tab["Y"] + ir_tab["N"]
    ir_tab["accuracy_pct"] = 100.0 * ir_tab["Y"] / ir_tab["assessable"].replace(0, np.nan)
    print(ir_tab.round(2).to_string())

    # --- 4. Derived region-perspective view (regions present in GT, n_gt >= 1) ---
    # Effective sampling requires a matched (TP) instance with a valid point/grid;
    # valid points of duplicates are not counted.
    _key = ["object_key", "perspective", "region"]
    _det = (_instm[_instm["match_type"].isin(["matched", "duplicate"])]
            .groupby(_key).size().rename("_det_flag"))
    _eff_edt  = (_instm[_instm["match_type"] == "matched"]
                 .groupby(_key)["edt_valid"].any().rename("eff_edt_ok"))
    _eff_grid = (_instm[_instm["match_type"] == "matched"]
                 .groupby(_key)["grid_valid"].any().rename("eff_grid_ok"))

    _rp = _region[_region["n_gt"].fillna(0) >= 1].copy()
    for c in ("detected",):
        _rp[c] = _rp[c].map({True: True, False: False, "True": True, "False": False}).fillna(False)
    _rp = _rp.merge(_det, on=_key, how="left").merge(_eff_edt, on=_key, how="left").merge(_eff_grid, on=_key, how="left")
    _rp["_det_flag"]  = _rp["_det_flag"].notna()
    _rp["eff_edt_ok"] = _rp["eff_edt_ok"].fillna(False)
    _rp["eff_grid_ok"] = _rp["eff_grid_ok"].fillna(False)

    _rp["ir_eff_edt"]  = [effective_ir_status(ic, s)
                          for ic, s in zip(_rp["ir_correct"], _rp["eff_edt_ok"])]
    _rp["ir_eff_grid"] = [effective_ir_status(ic, s)
                          for ic, s in zip(_rp["ir_correct"], _rp["eff_grid_ok"])]
    _rp["bucket_edt"]  = [pipeline_bucket(d, s, ic)
                          for d, s, ic in zip(_rp["detected"], _rp["eff_edt_ok"], _rp["ir_correct"])]
    _rp["bucket_grid"] = [pipeline_bucket(d, s, ic)
                          for d, s, ic in zip(_rp["detected"], _rp["eff_grid_ok"], _rp["ir_correct"])]
    _rp["pipeline_ok_edt"]  = _rp["bucket_edt"]  == "success"
    _rp["pipeline_ok_grid"] = _rp["bucket_grid"] == "success"

    # --- 5. Whole-pipeline success per region (gradeable denominator) ---
    def _success_summary(bucket_col):
        out = {}
        for region in VALID_REGIONS:
            sub = _rp[_rp["region"] == region]
            gradeable = sub[sub[bucket_col] != "no_ir_gt"]      # exclude ungradeable
            n = len(gradeable)
            ok = int((gradeable[bucket_col] == "success").sum())
            out[region] = (ok, n, _rate(ok, n))
        return out
    succ_edt  = _success_summary("bucket_edt")
    succ_grid = _success_summary("bucket_grid")
    pipe_df = pd.DataFrame([
        {"region": r,
         "edt_ok": succ_edt[r][0],  "edt_n": succ_edt[r][1],  "edt_pct": round(succ_edt[r][2], 1),
         "grid_ok": succ_grid[r][0], "grid_n": succ_grid[r][1], "grid_pct": round(succ_grid[r][2], 1)}
        for r in VALID_REGIONS]).set_index("region")
    print("\nWhole-pipeline success per region  [detected -> sampled -> IR; ungradeable excluded]")
    print("-" * 78)
    print(pipe_df.to_string())

    # --- 6. Loss decomposition (per region, per pipeline) ---
    def _loss_table(bucket_col):
        rows = []
        for region in VALID_REGIONS:
            sub = _rp[_rp["region"] == region]
            n = len(sub)
            counts = {b: int((sub[bucket_col] == b).sum()) for b in BUCKET_ORDER}
            row = {"region": region, "n_present": n}
            for b in BUCKET_ORDER:
                row[f"{b}_n"]   = counts[b]
                row[f"{b}_pct"] = round(_rate(counts[b], n), 1) if n else np.nan
            rows.append(row)
        return pd.DataFrame(rows).set_index("region")
    loss_edt  = _loss_table("bucket_edt")
    loss_grid = _loss_table("bucket_grid")
    print("\nLoss decomposition — EDT pipeline (% of present region-perspectives)")
    print("-" * 78)
    print(loss_edt[[f"{b}_pct" for b in BUCKET_ORDER]].to_string())
    print("\nLoss decomposition — Grid pipeline (% of present region-perspectives)")
    print("-" * 78)
    print(loss_grid[[f"{b}_pct" for b in BUCKET_ORDER]].to_string())

    # --- 7. Per-object rollup (share of present regions succeeding) ---
    def _obj_rollup(bucket_col):
        tmp = _rp[_rp[bucket_col] != "no_ir_gt"].copy()
        tmp["ok"] = (tmp[bucket_col] == "success").astype(float)
        return tmp.groupby("object_key")["ok"].mean().mul(100).round(1)
    obj_tab = pd.concat([_obj_rollup("bucket_edt").rename("edt_pct"),
                         _obj_rollup("bucket_grid").rename("grid_pct")], axis=1).fillna(0.0)
    print("\nPer-object whole-pipeline success (share of present region-perspectives)")
    print("-" * 78)
    print(obj_tab.to_string())

    # --- 8. Effective IR status per region ---
    def _eff_counts(col):
        return _rp.groupby("region")[col].value_counts().unstack(fill_value=0)
    print("\nEffective IR status per region (EDT pipeline: Y/N/--/NA)")
    print("-" * 78)
    print(_eff_counts("ir_eff_edt").reindex(VALID_REGIONS).fillna(0).astype(int).to_string())

    # --- Save aggregated and derived tables ---
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    seg.round(4).to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_segmentation_{_ts}.csv")
    if not samp.empty:
        samp.round(4).to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_sampling_{_ts}.csv")
    ir_tab.round(4).to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_ir_{_ts}.csv")
    pipe_df.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_pipeline_{_ts}.csv")
    loss_edt.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_loss_edt_{_ts}.csv")
    loss_grid.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_loss_grid_{_ts}.csv")
    obj_tab.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_agg_per_object_{_ts}.csv")
    _rp.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_derived_region_perspective_{_ts}.csv", index=False)
    print(f"\nSaved aggregated + derived tables to {RESULTS_DIR} (suffix _{_ts}).")

    AGG = {"seg": seg, "samp": samp, "ir": ir_tab, "pipe": pipe_df,
           "obj": obj_tab, "loss_edt": loss_edt, "loss_grid": loss_grid, "rp": _rp}


In [ ]:
# === Cell 11: Aggregation Plots ===

def _add_category_bands(ax, n):
    """Alternating light-grey / white vertical bands on a categorical x-axis."""
    for i in range(n):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
    ax.set_xlim(-0.5, n - 0.5)

def _value_labels(ax, xs, ys, fmt="{:.0f}", dy=1.0):
    """Bold value labels above bars; NaN values are skipped."""
    for x, y in zip(xs, ys):
        if y is None or (isinstance(y, float) and np.isnan(y)):
            continue
        ax.text(x, y + dy, fmt.format(y), ha="center", va="bottom",
                fontsize=_FONT_SIZE - 2, fontweight="bold")

if "AGG" not in globals() or _region.empty:
    print("Run Cell 10 first (no aggregated tables in memory).")
else:
    _ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    region_colors = [CLASS_COLORS_MPL[CLASS_MAP[r]] for r in VALID_REGIONS]
    n = len(VALID_REGIONS)
    x = np.arange(n)

    # --- Plot 1: segmentation precision / recall / F1 per region ---
    # TP/FP/FN counts are part of the x tick labels.
    seg = AGG["seg"]
    fig1, ax1 = plt.subplots(figsize=(_FW, _FW * 0.58))
    _add_category_bands(ax1, n)
    wbar = 0.26
    metric_c = {"precision": C_GT, "recall": C_GRID, "f1": C_EDT}
    for k, metric in enumerate(["precision", "recall", "f1"]):
        ax1.bar(x + (k - 1) * wbar, seg[metric].values, wbar,
                label=metric.capitalize(), color=metric_c[metric], zorder=3,
                edgecolor="0.25", linewidth=0.5)
    ax1.set_ylim(0, 1.0 * _YLIM_HEADROOM); ax1.set_ylabel("Score")
    ax1.set_title("Segmentation Detection Quality per Region")
    ax1.legend(loc="upper right")
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"{r}\nTP {int(seg.loc[r,'tp'])} / FP {int(seg.loc[r,'fp'])} / FN {int(seg.loc[r,'fn'])}"
                         for r in VALID_REGIONS])
    ax1.tick_params(axis="x", labelsize=_FONT_SIZE - 2)
    fig1.tight_layout()
    fig1.savefig(RESULTS_DIR / f"{PROJECT_NAME}_plot_segmentation_{_ts}.png", bbox_inches="tight")
    plt.show()

    # --- Plot 2: sampling validity EDT vs grid (matched instances) ---
    samp = AGG["samp"]
    if not samp.empty:
        fig2, ax2 = plt.subplots(figsize=(_FW, _FW * 0.55))
        _add_category_bands(ax2, n)
        wbar = 0.36
        e_vals = samp["edt_valid_pct"].values
        g_vals = samp["grid_valid_pct"].values
        ax2.bar(x - wbar/2, e_vals, wbar, label="EDT Point", color=C_EDT, zorder=3,
                edgecolor="0.25", linewidth=0.5)
        ax2.bar(x + wbar/2, g_vals, wbar, label="Grid", color=C_GRID, zorder=3,
                edgecolor="0.25", linewidth=0.5)
        ax2.set_xticks(x); ax2.set_xticklabels(VALID_REGIONS)
        ax2.set_ylim(0, 100 * _YLIM_HEADROOM)
        ax2.set_ylabel("Valid Matched Instances (%)")
        ax2.set_title("Sampling-Point Validity per Region")
        ax2.legend(loc="upper right")
        _value_labels(ax2, x - wbar/2, e_vals)
        _value_labels(ax2, x + wbar/2, g_vals)
        fig2.tight_layout()
        fig2.savefig(RESULTS_DIR / f"{PROJECT_NAME}_plot_sampling_{_ts}.png", bbox_inches="tight")
        plt.show()

    # --- Plot 3: IR accuracy per region (assessable object regions) ---
    ir_tab = AGG["ir"]
    fig3, ax3 = plt.subplots(figsize=(_FW, _FW * 0.55))
    _add_category_bands(ax3, n)
    acc = ir_tab["accuracy_pct"].values
    ax3.bar(x, acc, 0.6, color=region_colors, alpha=0.9, zorder=3,
            edgecolor="0.25", linewidth=0.5)
    ax3.set_xticks(x); ax3.set_xticklabels(VALID_REGIONS)
    ax3.set_ylim(0, 100 * _YLIM_HEADROOM); ax3.set_ylabel("IR Accuracy (%)")
    ax3.set_title("IR Classification Accuracy per Region")
    _value_labels(ax3, x, acc)
    for xi, r in enumerate(VALID_REGIONS):
        ax3.text(xi, 3, f"{ir_tab.loc[r,'Y']} / {int(ir_tab.loc[r,'assessable'])}",
                 ha="center", va="bottom", fontsize=_FONT_SIZE - 3, color="0.95")
    fig3.tight_layout()
    fig3.savefig(RESULTS_DIR / f"{PROJECT_NAME}_plot_ir_{_ts}.png", bbox_inches="tight")
    plt.show()

    # --- Plot 4: whole-pipeline success per region, EDT vs grid ---
    pipe = AGG["pipe"]
    fig4, ax4 = plt.subplots(figsize=(_FW, _FW * 0.55))
    _add_category_bands(ax4, n)
    wbar = 0.36
    pe = pipe["edt_pct"].values
    pg = pipe["grid_pct"].values
    ax4.bar(x - wbar/2, pe, wbar, label="Pipeline EDT",  color=C_EDT,  zorder=3,
            edgecolor="0.25", linewidth=0.5)
    ax4.bar(x + wbar/2, pg, wbar, label="Pipeline Grid", color=C_GRID, zorder=3,
            edgecolor="0.25", linewidth=0.5)
    ax4.set_xticks(x); ax4.set_xticklabels(VALID_REGIONS)
    ax4.set_ylim(0, 100 * _YLIM_HEADROOM)
    ax4.set_ylabel("Instances (%)")
    ax4.set_title("Whole-Pipeline Success per Region")
    ax4.legend(loc="upper right")
    _value_labels(ax4, x - wbar/2, pe)
    _value_labels(ax4, x + wbar/2, pg)
    fig4.tight_layout()
    fig4.savefig(RESULTS_DIR / f"{PROJECT_NAME}_plot_pipeline_{_ts}.png", bbox_inches="tight")
    plt.show()

    # --- Plots 5-6: loss decomposition (100 % stacked), one per pipeline ---
    def _plot_loss_stack(loss_tab, pipeline_name, fname):
        fig, ax = plt.subplots(figsize=(_FW, _FW * 0.58))
        _add_category_bands(ax, n)
        bottom = np.zeros(n)
        for b in BUCKET_ORDER:
            vals = loss_tab[f"{b}_pct"].values.astype(float)
            if np.nansum(vals) == 0:
                continue
            ax.bar(x, vals, 0.6, bottom=bottom, label=BUCKET_LABELS[b],
                   color=BUCKET_COLORS[b], zorder=3, edgecolor="0.25", linewidth=0.5)
            for xi in range(n):
                if vals[xi] >= 6:   # only annotate segments with room
                    ax.text(xi, bottom[xi] + vals[xi] / 2, f"{vals[xi]:.0f}",
                            ha="center", va="center", fontsize=_FONT_SIZE - 3,
                            color="white", fontweight="bold")
            bottom += np.nan_to_num(vals)
        ax.set_xticks(x); ax.set_xticklabels(VALID_REGIONS)
        ax.set_ylim(0, 100); ax.set_ylabel("Instances (%)")
        ax.set_title(f"Pipeline Outcome Decomposition per Region \u2014 {pipeline_name}")
        ax.legend(loc="upper right", fontsize=_FONT_SIZE - 3, ncol=1)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / fname, bbox_inches="tight")
        plt.show()

    _plot_loss_stack(AGG["loss_edt"],  "EDT Point",
                     f"{PROJECT_NAME}_plot_loss_stack_edt_{_ts}.png")
    _plot_loss_stack(AGG["loss_grid"], "Grid",
                     f"{PROJECT_NAME}_plot_loss_stack_grid_{_ts}.png")

    # --- Plots 7-8: loss grouped-bar comparison per region, per pipeline ---
    # Bars per region: Segmentation Loss, Sampling Loss, Classification Loss, Success.
    GROUPED_BUCKETS = ["no_detection", "no_sample", "ir_wrong", "success"]

    def _plot_loss_grouped(loss_tab, pipeline_name, fname):
        fig, ax = plt.subplots(figsize=(_FW, _FW * 0.58))
        _add_category_bands(ax, n)
        nb = len(GROUPED_BUCKETS)
        wbar = 0.8 / nb
        for k, b in enumerate(GROUPED_BUCKETS):
            vals = loss_tab[f"{b}_pct"].values.astype(float)
            offset = (k - (nb - 1) / 2) * wbar
            ax.bar(x + offset, vals, wbar * 0.92, label=BUCKET_LABELS[b],
                   color=BUCKET_COLORS[b], zorder=3, edgecolor="0.25", linewidth=0.5)
            _value_labels(ax, x + offset, vals)
        ax.set_xticks(x); ax.set_xticklabels(VALID_REGIONS)
        ax.set_ylim(0, 100 * _YLIM_HEADROOM)
        ax.set_ylabel("Instances (%)")
        ax.set_title(f"Pipeline Outcome Comparison per Region \u2014 {pipeline_name}")
        ax.legend(loc="upper right", fontsize=_FONT_SIZE - 2)
        fig.tight_layout()
        fig.savefig(RESULTS_DIR / fname, bbox_inches="tight")
        plt.show()

    _plot_loss_grouped(AGG["loss_edt"],  "EDT Point",
                       f"{PROJECT_NAME}_plot_loss_grouped_edt_{_ts}.png")
    _plot_loss_grouped(AGG["loss_grid"], "Grid",
                       f"{PROJECT_NAME}_plot_loss_grouped_grid_{_ts}.png")

    # --- Plot 9: combined EDT / Grid comparison ---
    # Segmentation Loss is pipeline-independent (single bar); Grid bars are dot-hatched.
    loss_edt  = AGG["loss_edt"]
    loss_grid = AGG["loss_grid"]

    n_bars = 5   # SegLoss, SampLoss-EDT, SampLoss-Grid, Success-EDT, Success-Grid
    wbar   = 0.8 / n_bars

    fig9, ax9 = plt.subplots(figsize=(_FW, _FW * 0.58))
    _add_category_bands(ax9, n)

    bar_specs = [
        # (label,                  data_source, metric_key,     color,                        hatch)
        ("Segmentation Loss",      loss_edt,   "no_detection", BUCKET_COLORS["no_detection"], ""),
        ("Sampling Loss \u2014 EDT",  loss_edt,   "no_sample",    BUCKET_COLORS["no_sample"],    ""),
        ("Sampling Loss \u2014 Grid", loss_grid,  "no_sample",    BUCKET_COLORS["no_sample"],    ".."),
        ("Success \u2014 EDT",        loss_edt,   "success",      BUCKET_COLORS["success"],      ""),
        ("Success \u2014 Grid",       loss_grid,  "success",      BUCKET_COLORS["success"],      ".."),
    ]

    for k, (lbl, src_tab, metric, color, hatch) in enumerate(bar_specs):
        vals   = src_tab[f"{metric}_pct"].values.astype(float)
        offset = (k - (n_bars - 1) / 2) * wbar
        ax9.bar(x + offset, vals, wbar * 0.92, label=lbl,
                color=color, hatch=hatch, zorder=3,
                edgecolor="0.25", linewidth=0.5)
        _value_labels(ax9, x + offset, vals)

    ax9.set_xticks(x); ax9.set_xticklabels(VALID_REGIONS)
    ax9.set_ylim(0, 100 * _YLIM_HEADROOM)
    ax9.set_ylabel("Instances (%)")
    ax9.set_title("Pipeline Outcome Comparison per Region")
    ax9.legend(loc="upper right", fontsize=_FONT_SIZE - 2)
    fig9.tight_layout()
    fig9.savefig(RESULTS_DIR / f"{PROJECT_NAME}_plot_combined_edt_grid_{_ts}.png",
                 bbox_inches="tight")
    plt.show()

    print(f"Saved 9 plots to {RESULTS_DIR} (suffix _{_ts}).")

In [ ]:
# === Cell 12: Pipeline Inspection Widget ===
# Browses any split with the held-out models; GT overlay and metrics are omitted for
# objects without segmentation GT. Metrics follow the Batch Analysis and Aggregation.

def _draw_dashed_polygon(ax, poly_px, color, dash=(6, 4)):
    """Draw a closed dashed polygon outline."""
    if poly_px is None or len(poly_px) < 3:
        return
    xs = list(poly_px[:, 0]) + [poly_px[0, 0]]
    ys = list(poly_px[:, 1]) + [poly_px[0, 1]]
    ax.plot(xs, ys, color=color, lw=1.4, ls=(0, dash), alpha=0.9)

# Status colours for the boxed labels.
_OK_COLOR  = C_EDT       # valid / true   (EDT_PEAK teal)
_BAD_COLOR = "#c44e52"   # invalid / false (red)
_NA_COLOR  = "#7f7f7f"   # NA / unknown   (grey)

def _boxed(ax, x, y, text, facecolor, fontsize=7):
    """Draw a rounded boxed label with white text."""
    ax.text(x, y, text, fontsize=fontsize, color="white", va="top", ha="left",
            zorder=8, bbox=dict(boxstyle="round,pad=0.2", fc=facecolor, ec="none", alpha=0.85))

def _render_inspection(object_key, per_idx, show_masks, show_contours,
                       show_edt, show_grid, show_gt, save_path=None):
    """Run the pipeline on one image and plot the selected overlay layers."""
    image_path = image_pool[object_key][per_idx]
    split = object_split[object_key]
    inst_recs, reg_recs, detail = analyze_image(
        image_path, object_key, split, return_detail=True)

    h, w = detail["h"], detail["w"]
    img_rgb = cv2.cvtColor(detail["result"].orig_img.copy(), cv2.COLOR_BGR2RGB).astype(float)

    # filled predicted masks
    if show_masks:
        overlay = img_rgb.copy()
        for region in VALID_REGIONS:
            rgb = CLASS_COLORS_RGB[CLASS_MAP[region]]
            for entry in detail["pred_mask_data"][region]:
                ys, xs = np.where(entry["raw"])
                overlay[ys, xs] = rgb
        img_rgb = img_rgb * (1 - MASK_ALPHA) + overlay * MASK_ALPHA

    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(img_rgb.astype(np.uint8))

    # predicted contours
    if show_contours:
        for region in VALID_REGIONS:
            mpl_c = CLASS_COLORS_MPL[CLASS_MAP[region]]
            for entry in detail["pred_mask_data"][region]:
                poly = entry["polygon_px"]
                if poly is None or len(poly) < 3:
                    continue
                xs = list(np.asarray(poly)[:, 0]) + [np.asarray(poly)[0, 0]]
                ys = list(np.asarray(poly)[:, 1]) + [np.asarray(poly)[0, 1]]
                ax.plot(xs, ys, color=mpl_c, lw=1.3, alpha=0.9)

    # dashed GT contours
    if show_gt and detail["has_gt"]:
        for region in VALID_REGIONS:
            gt_c = tuple(c / 255.0 for c in GT_COLORS_RGB[CLASS_MAP[region]])
            for entry in detail["gt_mask_data"][region]:
                _draw_dashed_polygon(ax, np.asarray(entry["polygon_px"]), gt_c)

    # grid hits (class-coloured small dots)
    if show_grid:
        by_class = defaultdict(lambda: ([], []))
        for gx, gy, cn, ii in detail["grid"]["hits"]:
            by_class[cn][0].append(gx); by_class[cn][1].append(gy)
        for cn, (gx, gy) in by_class.items():
            mpl_c = CLASS_COLORS_MPL[CLASS_MAP[cn]]
            ax.scatter(gx, gy, s=6, color=[mpl_c], marker="o", zorder=4, linewidths=0)

    # EDT sampling points ('x' marker: darkened class colour if valid, black if
    # invalid) with EDT / grid labels.
    for d in detail["draw_instances"]:
        mpl_c = CLASS_COLORS_MPL[CLASS_MAP[d["region"]]]
        dark  = tuple(max(0.0, c * 0.4) for c in mpl_c)   # darkened class colour
        edt_lbl  = f"EDT[{'valid' if d['edt_valid'] else 'invalid'}] s={d['edt_score']:.0f}"
        grid_lbl = f"Grid[{'valid' if d['grid_valid'] else 'invalid'}] n={d['n_grid_in_gt']}"
        if show_edt and d["edt_xy"] is not None:
            sx, sy = d["edt_xy"]
            mcol = dark if d["edt_valid"] else "#000000"
            ax.scatter([sx], [sy], marker="x", s=180, c=[mcol],
                       linewidths=3.0, zorder=6)
            _boxed(ax, sx + 6, sy - 6, edt_lbl,  _OK_COLOR if d["edt_valid"]  else _BAD_COLOR)
            _boxed(ax, sx + 6, sy + 20, grid_lbl, _OK_COLOR if d["grid_valid"] else _BAD_COLOR)

    # Per-instance segmentation label and per-region IR label
    _ir_drawn = set()
    reg_by_name = {r["region"]: r for r in reg_recs}
    for d in detail["draw_instances"]:
        poly = d["polygon_px"]
        if poly is None or len(np.asarray(poly)) < 3:
            continue
        poly = np.asarray(poly)
        ax0, ay0 = int(poly[:, 0].min()), int(poly[:, 1].min())
        iou_txt = "" if (d["iou_best"] is None or (isinstance(d["iou_best"], float) and np.isnan(d["iou_best"]))) \
                  else f" IoU={d['iou_best']:.2f}"
        _boxed(ax, ax0, max(0, ay0 - 4), f"{d['region']}[{d['match_type']}]{iou_txt}",
               CLASS_COLORS_MPL[CLASS_MAP[d["region"]]])
        # IR label once per region: raw class/confidence and effective status per
        # pipeline ("--" when the region cannot be sampled).
        if d["region"] not in _ir_drawn:
            _ir_drawn.add(d["region"])
            rr = reg_by_name[d["region"]]
            eff_edt  = effective_ir_status(rr["ir_correct"], bool(rr["sampling_edt_ok"]))
            eff_grid = effective_ir_status(rr["ir_correct"], bool(rr["sampling_grid_ok"]))
            _tok = {"Y": "true", "N": "false", "--": "--", "NA": "n/a"}
            conf = rr.get("ir_conf")
            conf_txt = "" if (conf is None or (isinstance(conf, float) and np.isnan(conf))) \
                       else f" c={float(conf):.2f}"
            if "N" in (eff_edt, eff_grid):
                ir_fc = _BAD_COLOR
            elif "Y" in (eff_edt, eff_grid):
                ir_fc = _OK_COLOR
            elif "--" in (eff_edt, eff_grid):
                ir_fc = C_GRID           # no valid sample
            else:
                ir_fc = _NA_COLOR
            _boxed(ax, ax0, max(0, ay0 - 4) + 20,
                   f"{rr['ir_pred']}{conf_txt} EDT[{_tok[eff_edt]}] Grid[{_tok[eff_grid]}]", ir_fc)

    # Title with per-region metric summary and pipeline bucket
    device_tag = f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
    _bkt = {"success": "OK", "ir_wrong": "IR", "no_sample": "SMP",
            "no_detection": "DET", "no_ir_gt": "NA"}
    lines = []
    for r in reg_recs:
        if r["has_gt"]:
            seg_s = f"TP={int(r['tp'])} FP={int(r['fp'])} FN={int(r['fn'])} dup={int(r['n_duplicates'])}"
        else:
            seg_s = "no seg-GT"
        present = r["has_gt"] and (r["n_gt"] >= 1)
        if present:
            be = pipeline_bucket(bool(r["detected"]), bool(r["sampling_edt_ok"]), r["ir_correct"])
            bg = pipeline_bucket(bool(r["detected"]), bool(r["sampling_grid_ok"]), r["ir_correct"])
            pipe_s = f"pipe EDT={_bkt[be]} Grid={_bkt[bg]}"
        else:
            pipe_s = "region absent"
        lines.append(f"{r['region']:5s} | {seg_s} | "
                     f"sample EDT={'Y' if r['sampling_edt_ok'] else 'N'} "
                     f"Grid={'Y' if r['sampling_grid_ok'] else 'N'} | "
                     f"IR:{r['ir_pred']}~GT:{r['ir_gt']}({r['ir_correct']}) | {pipe_s}")
    ax.set_title(
        f"{Path(image_path).name}  |  split={split}  |  "
        f"inference={detail['inference_ms']:.1f} ms ({device_tag})\n"
        f"solid=pred  dashed=GT  |  body=red cap=green label=blue  |  "
        f"x=EDT point (black=invalid)  dots=grid hits\n"
        f"pipe buckets: OK=success  IR=wrong material  SMP=no valid sample  "
        f"DET=not detected  NA=no IR GT\n" + "\n".join(lines),
        fontsize=8, loc="left")
    ax.axis("off")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved: {save_path}")
    plt.show()

# --- Widget controls ---
_split_options = [s for s in ("test", "cv", "unlisted")
                  if any(v == s for v in object_split.values())]
if not _split_options:
    _split_options = sorted(set(object_split.values()))

split_dropdown = widgets.Dropdown(
    options=[(s.upper(), s) for s in _split_options],
    value=(EVAL_SPLIT if EVAL_SPLIT in _split_options else _split_options[0]),
    description="Split:", layout=widgets.Layout(width="200px"))
object_dropdown = widgets.Dropdown(description="Object:", layout=widgets.Layout(width="330px"))
per_dropdown    = widgets.Dropdown(description="Perspective:",
                                   layout=widgets.Layout(width="220px"),
                                   style={"description_width": "initial"})

show_button   = widgets.Button(description="Show",   button_style="primary", icon="eye")
random_button = widgets.Button(description="Random", button_style="info",    icon="random")
save_button   = widgets.Button(description="Save PNG", icon="save")

show_masks_cb    = widgets.Checkbox(value=True,  description="Masks",    indent=False)
show_contours_cb = widgets.Checkbox(value=True,  description="Contours", indent=False)
show_edt_cb      = widgets.Checkbox(value=True,  description="EDT point", indent=False)
show_grid_cb     = widgets.Checkbox(value=False, description="Grid hits", indent=False)
show_gt_cb       = widgets.Checkbox(value=True,  description="GT (dashed)", indent=False)

viewer_output = widgets.Output()

def _objects_for_split(split):
    return sorted(k for k, s in object_split.items() if s == split)

def _refresh_objects(*_a):
    objs = _objects_for_split(split_dropdown.value)
    object_dropdown.options = objs
    object_dropdown.value = objs[0] if objs else None

def _refresh_pers(*_a):
    obj = object_dropdown.value
    pers = sorted(image_pool.get(obj, {}).keys()) if obj else []
    prev = per_dropdown.value
    per_dropdown.options = [(f"per_{p}", p) for p in pers]
    per_dropdown.value = prev if prev in pers else (pers[0] if pers else None)

split_dropdown.observe(lambda c: (_refresh_objects(), _refresh_pers()) if c["name"] == "value" else None)
object_dropdown.observe(lambda c: _refresh_pers() if c["name"] == "value" else None)
_refresh_objects(); _refresh_pers()

def _render(save_path=None):
    obj, per = object_dropdown.value, per_dropdown.value
    with viewer_output:
        clear_output(wait=True)
        if obj is None or per is None:
            print("No object / perspective available for this split.")
            return
        _render_inspection(obj, per, show_masks_cb.value, show_contours_cb.value,
                           show_edt_cb.value, show_grid_cb.value, show_gt_cb.value,
                           save_path=save_path)

def _on_show(b):   _render()
def _on_random(b):
    objs = _objects_for_split(split_dropdown.value)
    if not objs:
        return
    object_dropdown.value = random.choice(objs)
    pers = sorted(image_pool[object_dropdown.value].keys())
    if pers:
        per_dropdown.value = random.choice(pers)
    _render()
def _on_save(b):
    obj, per = object_dropdown.value, per_dropdown.value
    if obj is None or per is None:
        return
    sp = RESULTS_DIR / f"{PROJECT_NAME}_{split_dropdown.value}_{obj}_per_{per}.png"
    _render(save_path=sp)

show_button.on_click(_on_show)
random_button.on_click(_on_random)
save_button.on_click(_on_save)

display(
    widgets.VBox([
        widgets.HBox([split_dropdown, object_dropdown, per_dropdown]),
        widgets.HBox([show_button, random_button, save_button]),
        widgets.HBox([widgets.Label("Layers:"), show_masks_cb, show_contours_cb,
                      show_edt_cb, show_grid_cb, show_gt_cb]),
    ]),
    viewer_output,
)


In [ ]:
# === Cell 13: Pipeline Speed Benchmark ===
# Per-frame latency of segmentation, sampling and classification over N_REPEATS passes
# of the test-split images (after N_WARMUP discarded passes), for the Point (EDT_PEAK)
# and Grid sampling pipelines. Requires all cells up to Per-Image Pipeline Analysis.
# Timing:
#   Segmentation   - result.speed["inference"] (net YOLO inference, ms).
#   Sampling       - time.perf_counter() (summed EDT per instance, or grid_scan).
#   Classification - time.perf_counter() around predict() on one spectrum, shape
#                    (1, n_features); spectra are pre-loaded, disk I/O excluded.
#   std            - ddof=0 throughout.

# --- Config ---
N_WARMUP  = 5    # full passes over all test images, discarded
N_REPEATS = 30   # full passes over all test images, timed

# Stage colours taken from BUCKET_COLORS (no_detection, no_sample, ir_wrong, success).
_STAGE_COLORS = {
    "Segmentation":   "#4a4a4a",
    "Sampling":       C_GRID,          # "#e76f51"
    "Classification": "#c44e52",
    "Total":          C_EDT,           # "#2a9d8f"
}

# --- Pre-load test-split spectra (disk I/O excluded from timing) ---
_speed_spectra = {}   # (object_key, region) -> np.ndarray shape (1, n_features)
_n_loaded = 0
for _obj_key in eval_object_keys:
    for _region in VALID_REGIONS:
        _spath = spectra_index.get((_obj_key, _region))
        if _spath is not None:
            try:
                _vec = load_spectrum(_spath).reshape(1, -1)
                _speed_spectra[(_obj_key, _region)] = _vec
                _n_loaded += 1
            except Exception as _exc:
                print(f"  WARNING: could not pre-load spectrum {_spath.name}: {_exc}")
print(f"Pre-loaded {_n_loaded} test-split spectra into memory "
      f"({len(eval_object_keys)} objects × {len(VALID_REGIONS)} regions max).")

# --- Ordered (object_key, per_idx, image_path) list for the test split ---
_speed_images = []
for _obj_key in eval_object_keys:
    for _per_idx, _img_path in sorted(image_pool.get(_obj_key, {}).items()):
        _speed_images.append((_obj_key, _per_idx, _img_path))
_n_test_images = len(_speed_images)
print(f"Test images: {_n_test_images}")

if _n_test_images == 0:
    raise RuntimeError(f"No test images for EVAL_SPLIT='{EVAL_SPLIT}' — cannot benchmark.")

# --- Warmup: N_WARMUP full passes (all stages), results discarded ---
print(f"\nWarmup: {N_WARMUP} full passes over {_n_test_images} test images ...")
for _wu in range(N_WARMUP):
    for _obj_key, _per_idx, _img_path in _speed_images:
        _r = yolo_model_eval.predict(
            source=str(_img_path), conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
            imgsz=IMAGE_SIZE, device=DEVICE, retina_masks=True, max_det=MAX_DET,
            rect=True, verbose=False, save=False,
        )[0]
        _pi = yolo_result_to_instances(_r)
        _h, _w = _r.orig_img.shape[:2]
        _md = build_masks(_pi, _h, _w)
        for _reg in VALID_REGIONS:
            for _entry in _md[_reg]:
                if int(_entry["valid"].sum()) >= MIN_VALID_AREA_PX2:
                    compute_edt_peak(_entry["valid"])
        grid_scan(_md, GRID_SX_PX, GRID_SY_PX, GRID_ER_PX, _w, _h)
        for _reg in VALID_REGIONS:
            _k = (_obj_key, _reg)
            if _k in _speed_spectra:
                ir_classifier_eval.predict(_speed_spectra[_k])
    print(f"  warmup pass {_wu + 1}/{N_WARMUP} done")
print("Warmup complete.")

# --- Timed loop: N_REPEATS full passes ---
_timing_rows = []
print(f"\nTiming: {N_REPEATS} full passes ...")
for _rep in range(N_REPEATS):
    for _obj_key, _per_idx, _img_path in _speed_images:
        # --- Segmentation (net inference via Ultralytics internal timer) ---
        _result = yolo_model_eval.predict(
            source=str(_img_path), conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
            imgsz=IMAGE_SIZE, device=DEVICE, retina_masks=True, max_det=MAX_DET,
            rect=True, verbose=False, save=False,
        )[0]
        _seg_ms = float(_result.speed.get("inference", 0.0))

        # Build masks (not timed)
        _pred_inst = yolo_result_to_instances(_result)
        _h, _w = _result.orig_img.shape[:2]
        _pred_md = build_masks(_pred_inst, _h, _w)

        # --- Point sampling: EDT_PEAK per qualifying instance, summed ---
        _t0 = time.perf_counter()
        for _reg in VALID_REGIONS:
            for _entry in _pred_md[_reg]:
                if int(_entry["valid"].sum()) >= MIN_VALID_AREA_PX2:
                    compute_edt_peak(_entry["valid"])
        _point_ms = (time.perf_counter() - _t0) * 1000.0

        # --- Grid sampling: one whole-image grid_scan call ---
        _t0 = time.perf_counter()
        grid_scan(_pred_md, GRID_SX_PX, GRID_SY_PX, GRID_ER_PX, _w, _h)
        _grid_ms = (time.perf_counter() - _t0) * 1000.0

        # --- Classification: single-spectrum predict per present region ---
        _cls_ms = 0.0
        for _reg in VALID_REGIONS:
            _k = (_obj_key, _reg)
            if _k in _speed_spectra:
                _t0 = time.perf_counter()
                ir_classifier_eval.predict(_speed_spectra[_k])   # shape (1, n_features)
                _cls_ms += (time.perf_counter() - _t0) * 1000.0

        _timing_rows.append({
            "repeat": _rep,
            "object_key": _obj_key,
            "perspective": _per_idx,
            "seg_ms": _seg_ms,
            "point_sampling_ms": _point_ms,
            "grid_sampling_ms": _grid_ms,
            "classification_ms": _cls_ms,
        })
    if (_rep + 1) % 5 == 0 or _rep == N_REPEATS - 1:
        print(f"  repeat {_rep + 1}/{N_REPEATS} done")

speed_timing_df = pd.DataFrame(_timing_rows)
print(f"\nCollected {len(speed_timing_df)} timing rows "
      f"({N_REPEATS} repeats × {_n_test_images} images).")

# --- Cold-start check on the first timed image ---
_first_seg = speed_timing_df.iloc[0]["seg_ms"] if len(speed_timing_df) > 0 else 0.0
_seg_mean_all = speed_timing_df["seg_ms"].mean()
if _first_seg > 3 * _seg_mean_all:
    print(f"\n  NOTE: first timed image seg latency ({_first_seg:.2f} ms) is "
          f"{_first_seg / _seg_mean_all:.1f}x the overall mean ({_seg_mean_all:.2f} ms) "
          f"— likely a residual cold-start outlier despite warmup.")
else:
    print(f"\n  No cold-start outlier detected (first image: {_first_seg:.2f} ms, "
          f"mean: {_seg_mean_all:.2f} ms).")

# --- Aggregation: mean per image across all (image x repeat) pairs, ddof=0 ---
_speed_summary_rows = []
for _pipe, _samp_col in [("Point", "point_sampling_ms"),
                          ("Grid",  "grid_sampling_ms")]:
    _seg_mean  = speed_timing_df["seg_ms"].mean()
    _seg_std   = speed_timing_df["seg_ms"].std(ddof=0)
    _samp_mean = speed_timing_df[_samp_col].mean()
    _samp_std  = speed_timing_df[_samp_col].std(ddof=0)
    _cls_mean  = speed_timing_df["classification_ms"].mean()
    _cls_std   = speed_timing_df["classification_ms"].std(ddof=0)
    _tot_mean  = _seg_mean + _samp_mean + _cls_mean
    _tot_std   = float(np.sqrt(_seg_std**2 + _samp_std**2 + _cls_std**2))
    _fps       = 1000.0 / _tot_mean if _tot_mean > 0 else 0.0

    _speed_summary_rows.append({
        "pipeline":               _pipe,
        "seg_mean_ms":            round(_seg_mean, 4),
        "seg_std_ms":             round(_seg_std, 4),
        "sampling_mean_ms":       round(_samp_mean, 4),
        "sampling_std_ms":        round(_samp_std, 4),
        "classification_mean_ms": round(_cls_mean, 4),
        "classification_std_ms":  round(_cls_std, 4),
        "total_mean_ms":          round(_tot_mean, 4),
        "total_std_ms":           round(_tot_std, 4),
        "fps":                    round(_fps, 1),
    })

speed_summary_df = pd.DataFrame(_speed_summary_rows)

# --- Print summary ---
_device_tag = (f"GPU ({torch.cuda.get_device_name(0)})"
               if torch.cuda.is_available() else "CPU")
print("\n" + "=" * 72)
print("WHOLE-PIPELINE SPEED BENCHMARK SUMMARY")
print("=" * 72)
print(f"YOLO model:      {SEG_MODEL_PATH.name}")
print(f"IR classifier:   {IR_CLASSIFIER_MODEL_PATH.name}")
print(f"Test images:     {_n_test_images} (EVAL_SPLIT='{EVAL_SPLIT}')")
print(f"Pre-loaded spec: {_n_loaded}")
print(f"Warmup:          {N_WARMUP} passes   Timed: {N_REPEATS} passes")
print(f"Seg timing:      result.speed['inference'] (net inference only)")
print(f"Samp/Cls timing: time.perf_counter()   Std: ddof=0")
print(f"Device:          {_device_tag}")
print()
for _, _row in speed_summary_df.iterrows():
    print(f"  {_row['pipeline']:5s} pipeline:")
    print(f"    Segmentation:    {_row['seg_mean_ms']:8.3f} +/- {_row['seg_std_ms']:.3f} ms")
    print(f"    Sampling:        {_row['sampling_mean_ms']:8.3f} +/- {_row['sampling_std_ms']:.3f} ms")
    print(f"    Classification:  {_row['classification_mean_ms']:8.3f} +/- {_row['classification_std_ms']:.3f} ms")
    print(f"    {'─' * 43}")
    print(f"    Total:           {_row['total_mean_ms']:8.3f} +/- {_row['total_std_ms']:.3f} ms "
          f"({_row['fps']:.1f} FPS)")
    print()

# --- Save CSVs ---
_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
_raw_csv     = RESULTS_DIR / f"{PROJECT_NAME}_speed_raw_{_ts}.csv"
_summary_csv = RESULTS_DIR / f"{PROJECT_NAME}_speed_summary_{_ts}.csv"
speed_timing_df.to_csv(_raw_csv, index=False)
speed_summary_df.to_csv(_summary_csv, index=False)

LATEST_SPEED_RAW_CSV     = _raw_csv
LATEST_SPEED_SUMMARY_CSV = _summary_csv

print(f"Saved:")
print(f"  Raw timings: {_raw_csv.name}  ({len(speed_timing_df)} rows)")
print(f"  Summary:     {_summary_csv.name}  ({len(speed_summary_df)} rows)")

In [ ]:
# === Cell 14: Speed Benchmark Plot ===
# Grouped bars per pipeline (Point, Grid): Segmentation, Sampling, Classification,
# Total. Reloads the summary from LATEST_SPEED_SUMMARY_CSV.

from matplotlib.transforms import blended_transform_factory as _blend_tf

# --- Reload summary from CSV ---
_speed_sum = pd.read_csv(LATEST_SPEED_SUMMARY_CSV)

# Stage colours (as in the Speed Benchmark cell)
_STAGE_COLORS = {
    "Segmentation":   "#4a4a4a",       # no_detection
    "Sampling":       "#e76f51",       # no_sample  (C_GRID)
    "Classification": "#c44e52",       # ir_wrong
    "Total":          "#2a9d8f",       # success    (C_EDT)
}

_STAGES    = ["Segmentation", "Sampling", "Classification", "Total"]
_PIPELINES = ["Point", "Grid"]

# Build bar data: (pipeline, stage) -> (mean, std)
_bar_data = {}
for _, _row in _speed_sum.iterrows():
    _p = _row["pipeline"]
    _bar_data[(_p, "Segmentation")]   = (_row["seg_mean_ms"],            _row["seg_std_ms"])
    _bar_data[(_p, "Sampling")]       = (_row["sampling_mean_ms"],       _row["sampling_std_ms"])
    _bar_data[(_p, "Classification")] = (_row["classification_mean_ms"], _row["classification_std_ms"])
    _bar_data[(_p, "Total")]          = (_row["total_mean_ms"],          _row["total_std_ms"])

# --- Positions: 4 bars per group, gap between groups ---
_n_stages = len(_STAGES)
_bar_w    = 0.72
_gap      = 1.8          # extra spacing between pipeline groups

_positions, _means, _stds, _colors = [], [], [], []
_group_centers = []
for _gi, _pipe in enumerate(_PIPELINES):
    _gc_pos = []
    for _si, _stage in enumerate(_STAGES):
        _x = _gi * (_n_stages + _gap) + _si
        _positions.append(_x)
        _gc_pos.append(_x)
        _m, _s = _bar_data[(_pipe, _stage)]
        _means.append(_m)
        _stds.append(_s)
        _colors.append(_STAGE_COLORS[_stage])
    _group_centers.append(np.mean(_gc_pos))

# --- Plot ---
fig, ax = plt.subplots(figsize=(_FW, _FW * 0.52))

ax.bar(_positions, _means, width=_bar_w, color=_colors, edgecolor="white",
       lw=0.5, yerr=_stds, capsize=3, error_kw={"lw": 1.0, "capthick": 1.0},
       zorder=3)

# Alternating background bands per pipeline group
for _gi in range(len(_PIPELINES)):
    _x0 = _gi * (_n_stages + _gap) - _bar_w / 2 - 0.35
    _x1 = _gi * (_n_stages + _gap) + _n_stages - 1 + _bar_w / 2 + 0.35
    if _gi % 2 == 0:
        ax.axvspan(_x0, _x1, color="#f0f0f0", zorder=0)

# y-axis headroom for bold value labels
_ymax = max(_m + _s for _m, _s in zip(_means, _stds))
ax.set_ylim(0, _ymax * _YLIM_HEADROOM)

# Bold value labels on each bar; Total bars also show FPS
for _i, (_x, _m, _s) in enumerate(zip(_positions, _means, _stds)):
    _gi = _i // _n_stages
    _si = _i % _n_stages
    _pipe = _PIPELINES[_gi]
    _stage = _STAGES[_si]

    if _stage == "Total":
        _fps = _speed_sum[_speed_sum["pipeline"] == _pipe]["fps"].values[0]
        _label = f"{_m:.2f}\n({_fps:.0f} FPS)"
    else:
        _label = f"{_m:.2f}"
    ax.text(_x, _m + _s + _ymax * 0.012, _label, ha="center", va="bottom",
            fontsize=_FONT_SIZE - 1, fontweight="bold")

# x-axis: stage labels under each bar
ax.set_xticks(_positions)
ax.set_xticklabels([])

# Pipeline group labels (data-x, axes-y via blended transform)
_trans = _blend_tf(ax.transData, ax.transAxes)
for _gc, _pipe in zip(_group_centers, _PIPELINES):
    ax.text(_gc, -0.08, f"{_pipe} Pipeline", transform=_trans,
            ha="center", va="top", fontsize=_FONT_SIZE, fontweight="bold",
            clip_on=False)

ax.set_ylabel("Latency (ms)")

# Legend — one entry per stage
_handles = [mpatches.Patch(facecolor=_STAGE_COLORS[s], label=s) for s in _STAGES]
ax.legend(handles=_handles, loc="upper left")

fig.subplots_adjust(bottom=0.15)
fig.tight_layout()

# Save
_speed_plot_path = RESULTS_DIR / f"{PROJECT_NAME}_speed_benchmark.png"
fig.savefig(_speed_plot_path, bbox_inches="tight", dpi=150)
print(f"Saved: {_speed_plot_path}")
plt.show()